In [1]:
import gzip
import json
import os
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
filepath = os.path.join("..", "data", "c4-train.00000-of-01024-30K.json.gz")

documents = []

with gzip.open(filepath, "rt", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        if "text" in data:
            documents.append(data["text"])

print("Number of documents:", len(documents))

Number of documents: 30000


In [3]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for",
    "from", "has", "have", "he", "her", "his", "i", "in",
    "is", "it", "its", "of", "on", "or", "that", "the",
    "their", "this", "to", "was", "were", "will", "with",
    "you", "your"
}

def pipeline_a(text):
    # Lowercase + whitespace tokenization
    return text.lower().split()


def pipeline_b(text):
    # Lowercase
    text = text.lower()

    # Punctuation normalization:
    # thay punctuation bằng khoảng trắng
    text = re.sub(r"[^\w\s]", " ", text)

    # Tokenization
    tokens = text.split()

    # Stopword handling
    tokens = [token for token in tokens if token not in STOPWORDS]

    return tokens


def pipeline_c_text(text):
    # Normalization
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
sample = "The medical-image model, however, works very well!"

print("Original:")
print(sample)

print("\nPipeline A:")
print(pipeline_a(sample))

print("\nPipeline B:")
print(pipeline_b(sample))

print("\nPipeline C normalized:")
print(pipeline_c_text(sample))

Original:
The medical-image model, however, works very well!

Pipeline A:
['the', 'medical-image', 'model,', 'however,', 'works', 'very', 'well!']

Pipeline B:
['medical', 'image', 'model', 'however', 'works', 'very', 'well']

Pipeline C normalized:
the medical image model however works very well


In [6]:
#TF-IDF cho cả A và B
texts_a = [" ".join(pipeline_a(text)) for text in documents]
texts_b = [" ".join(pipeline_b(text)) for text in documents]

vectorizer_a = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

vectorizer_b = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

X_a = vectorizer_a.fit_transform(texts_a)
X_b = vectorizer_b.fit_transform(texts_b)

print("Pipeline A shape:", X_a.shape)
print("Pipeline B shape:", X_b.shape)

Pipeline A shape: (30000, 473388)
Pipeline B shape: (30000, 193803)


In [7]:
# Pipepile C
texts_c = [pipeline_c_text(text) for text in documents]

vectorizer_c = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5)
)

X_c = vectorizer_c.fit_transform(texts_c)

print("Pipeline C shape:", X_c.shape)

Pipeline C shape: (30000, 1104843)


In [8]:
vocab_a = len(vectorizer_a.vocabulary_)
vocab_b = len(vectorizer_b.vocabulary_)
vocab_c = len(vectorizer_c.vocabulary_)

print("Vocabulary size")
print("A:", vocab_a)
print("B:", vocab_b)
print("C:", vocab_c)

Vocabulary size
A: 473388
B: 193803
C: 1104843


In [9]:
avg_tokens_a = np.mean([len(pipeline_a(text)) for text in documents])
avg_tokens_b = np.mean([len(pipeline_b(text)) for text in documents])

# C: số character n-grams sinh ra cho mỗi document
def count_char_ngrams(text, ngram_range=(3, 5)):
    total = 0
    for n in range(ngram_range[0], ngram_range[1] + 1):
        total += max(0, len(text) - n + 1)
    return total

avg_tokens_c = np.mean([
    count_char_ngrams(text)
    for text in texts_c
])

print("Average tokens/features per document")
print("A:", avg_tokens_a)
print("B:", avg_tokens_b)
print("C:", avg_tokens_c)

Average tokens/features per document
A: 361.08906666666667
B: 250.88393333333335
C: 6283.5126


In [10]:
def matrix_sparsity(X):
    N, V = X.shape
    return 1 - X.nnz / (N * V)


sparsity_a = matrix_sparsity(X_a)
sparsity_b = matrix_sparsity(X_b)
sparsity_c = matrix_sparsity(X_c)

print("Matrix sparsity")
print("A:", sparsity_a)
print("B:", sparsity_b)
print("C:", sparsity_c)

Matrix sparsity
A: 0.9996113415774516
B: 0.9992168406405818
C: 0.9972306317730212


In [ ]:
split_index = int(len(documents) * 0.8) # 80%

train_docs = documents[:split_index]
test_docs = documents[split_index:]


def word_oov_rate(train_docs, test_docs, pipeline):
    train_vocab = set()

    for text in train_docs:
        train_vocab.update(pipeline(text))

    total = 0
    oov = 0

    for text in test_docs:
        tokens = pipeline(text)

        for token in tokens:
            total += 1
            if token not in train_vocab:
                oov += 1

    return oov / total if total else 0


oov_a = word_oov_rate(train_docs, test_docs, pipeline_a)
oov_b = word_oov_rate(train_docs, test_docs, pipeline_b)

print("OOV rate")
print("A:", oov_a)
print("B:", oov_b)

OOV rate
A: 0.03973201253798483
B: 0.02623190252942615


In [ ]:
# Tỷ lệ character n-gram trong test không xuất hiện trong vocabulary train.
def char_ngram_set(text, ngram_range=(3, 5)):
    grams = set()

    for n in range(ngram_range[0], ngram_range[1] + 1):
        for i in range(len(text) - n + 1):
            grams.add(text[i:i+n])

    return grams


train_char_vocab = set()

for text in [pipeline_c_text(t) for t in train_docs]:
    train_char_vocab.update(char_ngram_set(text))


total_grams = 0
oov_grams = 0

for text in [pipeline_c_text(t) for t in test_docs]:
    grams = char_ngram_set(text)

    for gram in grams:
        total_grams += 1
        if gram not in train_char_vocab:
            oov_grams += 1

oov_c = oov_grams / total_grams if total_grams else 0

print("C feature OOV rate:", oov_c)

C feature OOV rate: 0.005965753348042551


In [15]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]


def search(query, vectorizer, X, preprocess, top_k=5):
    query_text = " ".join(preprocess(query))

    q = vectorizer.transform([query_text])
    scores = cosine_similarity(q, X).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        (int(i), float(scores[i]))
        for i in top_indices
    ]


for query in queries:
    print("QUERY:", query)

    print("\nPipeline A:")
    for idx, score in search(
        query, vectorizer_a, X_a, pipeline_a
    ):
        print(idx, round(score, 4), documents[idx][:150].replace("\n", " "))

    print("\nPipeline B:")
    for idx, score in search(
        query, vectorizer_b, X_b, pipeline_b
    ):
        print(idx, round(score, 4), documents[idx][:150].replace("\n", " "))

QUERY: medical image classification

Pipeline A:
18971 0.3863 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in
27781 0.2333 ❶Press Officer Resume Sample. Based on your requirements and skills you can shuffle these sections in the resume. Consults reference books and materia
190 0.2283 This title is a comprehensive account of the key aspects of medical leadership. A highly accessible, text book-style resource, it explores how the med
17794 0.2245 Filters the output of 'wp_calculate_image_sizes()'. A source size value for use in a 'sizes' attribute. Requested size. Image size or array of width a
8370 0.2239 What is a Online Medical Second Opinion? For over 25 years, patients and families have looked to We Care India for medical advice and treatment. Now y

Pipeline B:
18971 0.4211 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning cons

In [16]:
for query in queries:
    print("QUERY:", query)

    query_text = pipeline_c_text(query)
    q = vectorizer_c.transform([query_text])

    scores = cosine_similarity(q, X_c).flatten()
    top_indices = np.argsort(scores)[::-1][:5]

    for idx in top_indices:
        print(
            idx,
            round(float(scores[idx]), 4),
            documents[idx][:150].replace("\n", " ")
        )

QUERY: medical image classification
18971 0.3583 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in
8527 0.3443 History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico.
15284 0.2 Citation: Wu, T., Armstrong, P.R., Maghirang, E.B. 2018. Vis- and NIR-based instruments for detection of black-tip damaged wheat kernels: A comparativ
23196 0.1961 With over 200 free vector images, Vecteezy is updated frequently with new graphics. Vector images can be searched & sorted by file type (Adobe Illustr
11625 0.1632 The problem I am tackling is categorizing short texts into multiple classes. My current approach is to use tf-idf weighted term frequencies and learn 
QUERY: transformer language model
25428 0.3736 Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses wha

In [17]:
print(f"{'Metric':<35} {'A':>15} {'B':>15} {'C':>15}")
print("-" * 85)

print(f"{'Vocabulary size':<35} {vocab_a:>15} {vocab_b:>15} {vocab_c:>15}")
print(f"{'Average tokens/features per doc':<35} {avg_tokens_a:>15.2f} {avg_tokens_b:>15.2f} {avg_tokens_c:>15.2f}")
print(f"{'Matrix sparsity':<35} {sparsity_a:>15.6f} {sparsity_b:>15.6f} {sparsity_c:>15.6f}")
print(f"{'OOV rate':<35} {oov_a:>15.6f} {oov_b:>15.6f} {oov_c:>15.6f}")

Metric                                            A               B               C
-------------------------------------------------------------------------------------
Vocabulary size                              473388          193803         1104843
Average tokens/features per doc              361.09          250.88         6283.51
Matrix sparsity                            0.999611        0.999217        0.997231
OOV rate                                   0.039732        0.026232        0.005966


1. Lowercasing làm thay đổi vocabulary như thế nào?

Lowercasing gộp về chữ hoa/chữ thường thành cùng một token. Vì vậy vocabulary thường giảm hoặc thay đổi cấu trúc.

2. Stopword removal có luôn cải thiện representation không?

Không nhất thiết. Stopword removal làm giảm vocabulary và có thể giảm số feature không mang nhiều thông tin phân biệt. Nhưng stopword đôi khi mang thông tin ngữ pháp hoặc ý nghĩa.

3. Loại punctuation có thể làm mất thông tin gì?

Có thể mất các tín hiệu như:
- dấu câu biểu thị cấu trúc câu
- dấu `?`, `!` biểu thị sắc thái
- ký hiệu hoặc định dạng đặc biệt
- một số token có dấu gạch nối hoặc ký hiệu có ý nghĩa

Do đó punctuation normalization có lợi cho vocabulary nhưng có thể làm mất thông tin.

4. Pipeline nào tạo sparse matrix nhất?

Pipeline A

5. Pipeline nào cho search tốt nhất?

Pipepile C

6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?

Không. Vocabulary size chỉ là một đặc tính của representation. Search phụ thuộc vào tokenization, TF-IDF weighting, lexical overlap và mức độ phù hợp giữa query với document. Vì vậy vocabulary nhỏ hơn không tự động đồng nghĩa search tốt hơn.